# Lakehouse Maintenance — Configure Table Properties

Standalone maintenance notebook for applying Delta table property settings
across all Delta tables in a target Lakehouse.

| Property | What It Controls | Cost of Re-Running |
|----------|------------------|--------------------|
| **delta.enableChangeDataFeed** | Tracks row-level INSERT/UPDATE/DELETE for downstream CDC consumption. | Near-zero — skipped when already set. |
| **delta.autoOptimize.autoCompact** | Auto-compacts small files after writes to prevent fragmentation. | Near-zero — skipped when already set. |
| **delta.deletedFileRetentionDuration** | How long VACUUM keeps tombstoned files before physical deletion. | Near-zero — skipped when already set. |

**Schedule:** One-time or occasional. Re-run after adding new tables to apply properties.

**Schema-aware:** Discovers tables across all schemas in a schema-enabled Lakehouse.

**Idempotent:** Reads each table's current properties first and skips tables already at the target values.

## Configuration

Target workspace and Lakehouse are auto-detected from the attached Lakehouse — override the constants below to point at a different environment. `SCHEMAS` scopes the run to a subset of schemas (empty = all). `TABLE_PROPERTIES` declares the properties to apply: add an entry to apply it, remove an entry to leave that property untouched on all tables.

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────

context = notebookutils.runtime.context

# --- Lakehouse Target ---
# Auto-detected from the notebook's attached lakehouse.
# Override these if targeting a different workspace or lakehouse.
WORKSPACE_NAME = context.get("currentWorkspaceName")
LAKEHOUSE_NAME = context.get("defaultLakehouseName")

# --- Schemas ---
# [] = scan all schemas in the Lakehouse (schema-enabled or default `dbo`).
# ["salesforce", "netsuite"] = only scan these schemas (case-sensitive match
# against the folder names under `.../Lakehouse/Tables/`).
SCHEMAS = []

# --- Table Limit ---
# -1 = process all discovered tables.
# Positive integer = cap for testing (e.g., 5 to validate on a subset).
TABLE_LIMIT = -1

# --- V-Order Target Tables ---
# V-Order is disabled by default in new Fabric workspaces. It adds ~15% write
# overhead but gives 40-60% cold-cache read improvement for Direct Lake and
# Power BI consumers. Only blanket-apply to gold-layer / reporting tables —
# never to bronze/silver staging or high-churn tables.
# Entries must match the backticked form produced by discovery:
#   "`schema`.`table`"
VORDER_TABLES = [
    # "`gold`.`FactSales`",
    # "`gold`.`DimCustomer`",
]

# --- Table Properties ---
# Map of TBLPROPERTIES key -> target value (both strings).
# Each table's current value is read first; only differing tables are altered.
# To leave a property untouched, remove its entry from the dict.
TABLE_PROPERTIES = {
    "delta.enableChangeDataFeed":         "true",
    "delta.autoOptimize.autoCompact":     "true",
    # optimizeWrite pairs with autoCompact: autoCompact rewrites small files
    # AFTER commit; optimizeWrite sizes them BEFORE commit. Using both avoids
    # intermediate small-file churn in the Delta log.
    "delta.autoOptimize.optimizeWrite":   "true",
    "delta.deletedFileRetentionDuration": "interval 90 days",
    # logRetentionDuration MUST be >= deletedFileRetentionDuration. If the log
    # is pruned before the data files it references, time travel breaks even
    # though the underlying Parquet files still exist on disk.
    "delta.logRetentionDuration":         "interval 90 days",
}

# --- Derived Path (do not edit) ---
SOURCE_ROOT = (
    f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_NAME}.Lakehouse/Tables"
)

print("--- Configuration ---")
print(f"  Workspace:        {WORKSPACE_NAME}")
print(f"  Lakehouse:        {LAKEHOUSE_NAME}")
print(f"  Schemas:          {'ALL' if not SCHEMAS else ', '.join(SCHEMAS)}")
print(f"  Table limit:      {'ALL' if TABLE_LIMIT == -1 else TABLE_LIMIT}")
print(f"  V-Order tables:   {len(VORDER_TABLES)}")
print(f"  Properties:       {len(TABLE_PROPERTIES)}")
for _k, _v in TABLE_PROPERTIES.items():
    print(f"    - {_k} = {_v}")
print(f"  Source root:      {SOURCE_ROOT}")

### Session vs. Table Property Precedence

`delta.autoOptimize.autoCompact` and `delta.autoOptimize.optimizeWrite` are **transient** properties — they can be overridden by session-level Spark configs (`spark.databricks.delta.autoCompact.enabled`, `spark.databricks.delta.optimizeWrites.enabled`). When both are set, **session wins**.

Setting them at the table level (here) is the right place for consistency across writers: every job that writes to the table — notebooks, pipelines, external Spark — sees the same behaviour regardless of the session config it happens to inherit.

If `nb_spark_config` also sets the session-level equivalents, either keep the values in agreement or accept that session-level will win for jobs that run through that config.

## Discover Delta Tables

Scans schemas under the Lakehouse Tables path — all schemas by default, or only those listed in `SCHEMAS`. Validates each directory contains a `_delta_log` folder before including it — non-Delta folders are skipped silently. Schemas named in `SCHEMAS` but missing from the Lakehouse produce a warning.

In [ ]:
from notebookutils import mssparkutils
import py4j
import datetime

all_tables = []

try:
    schemas = mssparkutils.fs.ls(SOURCE_ROOT)
    discovered_schema_names = {s.name for s in schemas if s.isDir}

    # Warn on any requested schema that doesn't exist under the Lakehouse
    if SCHEMAS:
        missing = [s for s in SCHEMAS if s not in discovered_schema_names]
        if missing:
            print(f"  ⚠️  Requested schemas not found: {', '.join(missing)}")

    for schema in schemas:
        if not schema.isDir:
            continue

        schema_name = schema.name

        # Apply the SCHEMAS filter when populated
        if SCHEMAS and schema_name not in SCHEMAS:
            continue

        print(f"  Scanning schema: {schema_name}")

        try:
            tables = mssparkutils.fs.ls(schema.path)
        except Exception as e:
            print(f"  ⚠️  Could not scan schema '{schema_name}': {e}")
            continue

        for t in tables:
            if not t.isDir:
                continue

            # Confirm Delta table by checking for _delta_log directory
            delta_log_path = f"{t.path}/_delta_log"
            try:
                mssparkutils.fs.ls(delta_log_path)
                spark_table_name = f"`{schema_name}`.`{t.name}`"
                all_tables.append(spark_table_name)
            except py4j.protocol.Py4JJavaError:
                # Expected for non-Delta folders — skip silently
                pass
            except Exception as e:
                print(f"  ⚠️  Error checking {schema_name}/{t.name}: {e}")

except Exception as e:
    print(f"❌ FATAL: Could not list source root '{SOURCE_ROOT}': {e}")
    raise

scope = "all schemas" if not SCHEMAS else f"schemas: {', '.join(SCHEMAS)}"
print(f"\nDiscovered {len(all_tables)} Delta tables across {scope}.")

## Apply Properties

For each discovered table: reads current `TBLPROPERTIES` via `DESCRIBE DETAIL`, then issues `ALTER TABLE … SET TBLPROPERTIES` only for properties whose value differs from the target. Properties already at the target value are skipped with no ALTER issued — this keeps re-runs cheap and avoids unnecessary Delta log commits.

In [ ]:
if TABLE_LIMIT == -1:
    tables_to_process = all_tables
else:
    tables_to_process = all_tables[:TABLE_LIMIT]

print(f"Processing {len(tables_to_process)} tables.\n")

results = []         # Per-table outcome tracking
failed_tables = []   # Tables where any property failed

start_time = datetime.datetime.now()

for table_name in tables_to_process:
    print(f"── {table_name} ──")
    result = {
        "table": table_name,
        "changed": [],
        "already_set": [],
        "failed": [],
    }

    # Read current properties once per table to avoid redundant ALTERs
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()
        current_props = detail[0].asDict().get("properties", {}) or {}
    except Exception as e:
        print(f"  ❌ Could not read table detail: {e}")
        result["failed"].append("DESCRIBE DETAIL")
        failed_tables.append(table_name)
        results.append(result)
        continue

    # Build the target property set for this table: base + V-Order opt-in
    table_targets = dict(TABLE_PROPERTIES)
    if table_name in VORDER_TABLES:
        table_targets["delta.parquet.vorder.enabled"] = "true"

    for prop_name, target_value in table_targets.items():
        current_value = current_props.get(prop_name)

        if current_value == target_value:
            print(f"  🔹 {prop_name} = {target_value} (already set)")
            result["already_set"].append(prop_name)
            continue

        try:
            spark.sql(
                f"ALTER TABLE {table_name} "
                f"SET TBLPROPERTIES ('{prop_name}' = '{target_value}')"
            )
            print(f"  ✅ {prop_name} → {target_value}")
            result["changed"].append(prop_name)
        except Exception as e:
            print(f"  ❌ {prop_name}: {e}")
            result["failed"].append(prop_name)
            if table_name not in failed_tables:
                failed_tables.append(table_name)

    results.append(result)

end_time = datetime.datetime.now()
elapsed = end_time - start_time

## Summary

In [ ]:
total_changed     = sum(len(r["changed"])     for r in results)
total_already_set = sum(len(r["already_set"]) for r in results)
total_failed      = sum(len(r["failed"])      for r in results)

print("\n" + "=" * 80)
print("  CONFIGURE SUMMARY")
print("=" * 80)
print(f"\n  Tables processed:   {len(results)}")
print(f"  Properties changed: {total_changed}")
print(f"  Already at target:  {total_already_set}")
print(f"  Failed:             {total_failed}")
print(f"  Elapsed time:       {elapsed}")

print(f"\n  {'Table':<45} {'Changed':>9} {'AlreadySet':>11} {'Failed':>9}")
print(f"  {'-'*45} {'-'*9} {'-'*11} {'-'*9}")

for r in results:
    print(
        f"  {r['table']:<45} "
        f"{len(r['changed']):>9} "
        f"{len(r['already_set']):>11} "
        f"{len(r['failed']):>9}"
    )

if failed_tables:
    print(f"\n  ⚠️  FAILED TABLES ({len(failed_tables)}):")
    for t in failed_tables:
        r = next((x for x in results if x["table"] == t), None)
        detail = ", ".join(r["failed"]) if r and r["failed"] else "unknown"
        print(f"    - {t}: {detail}")
else:
    print(f"\n  ✅ All tables completed successfully.")

print("\n" + "=" * 80)